# Post-Quantum Cryptography on FPGA
## Hardware-Accelerated Polynomial Multiplication for CRYSTALS-Kyber

---

**The short version:** quantum computers will break the encryption protecting your bank, your messages, and your identity.
The world is switching to *post-quantum cryptography* — but the new math is expensive.
This demo shows a custom FPGA chip that makes the hardest part **32× faster**.

No cryptography background needed.

In [2]:
%matplotlib inline
import sys, os, struct, hashlib, time, subprocess
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

BASE = '/home/xilinx/jupyter_notebooks/kyber-ntt-fpga'
sys.path.insert(0, os.path.join(BASE, 'ps'))
sys.path.insert(0, os.path.join(BASE, 'golden'))

from kyber_kem import run_kem, ntt_mul_sw, ntt_mul_hw, KyberKEM

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

SW_COLOR = '#4878CF'
HW_COLOR = '#E84D3D'

print('Setup complete.')

ModuleNotFoundError: No module named 'matplotlib'

---
## Part 1 — The Problem

### Why is encryption about to break?

Today's internet encryption (RSA, ECDH) relies on math problems that are easy to check but hard to solve — like factoring a huge number into its primes. A classical computer would take longer than the age of the universe.

A **quantum computer** running Shor's algorithm can solve the same problem in hours.

NIST standardized **CRYSTALS-Kyber** (FIPS 203) as the replacement. Instead of factoring, it hides secrets inside *polynomials over a lattice* — a completely different mathematical structure that quantum computers cannot attack efficiently.

### How Kyber works

Alice and Bob want to agree on a secret key without ever meeting. Kyber works like a padlock:
Alice generates a padlock (public key) and sends it to Bob. Bob puts the secret inside and snaps the padlock shut (encapsulation). Only Alice has the key to open it (decapsulation). Both end up with the same shared secret — and an eavesdropper learns nothing.

The padlock and key are built from **polynomials** — lists of 256 numbers between 0 and 3328. Locking and unlocking requires multiplying these polynomials together. **That multiply is the bottleneck.**

In [1]:
# --- Visual: what is a polynomial in Kyber? ---
rng = np.random.default_rng(7)
a_demo = rng.integers(0, 3329, 256)
b_demo = rng.integers(0, 3329, 256)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))

for ax, poly, color, label in zip(
    axes,
    [a_demo, b_demo, None],
    [SW_COLOR, '#2ca02c', HW_COLOR],
    ['Polynomial  a  (public key component)',
     'Polynomial  b  (message)',
     'Result  c = a × b  mod (x²⁵⁶+1, 3329)']
):
    if poly is None:
        ax.text(0.5, 0.5, '?\n\n(computed by\nthe FPGA)',
                ha='center', va='center', fontsize=14,
                color=color, fontweight='bold', transform=ax.transAxes)
        ax.set_facecolor('#fff8f8')
    else:
        ax.bar(range(256), poly, width=1.0, color=color, alpha=0.75, linewidth=0)
        ax.set_ylim(0, 3329)
        ax.set_ylabel('Coefficient value (0–3328)')
    ax.set_xlabel('Coefficient index (0–255)')
    ax.set_title(label, fontsize=10, fontweight='bold')

fig.suptitle('A Kyber polynomial: 256 numbers, each between 0 and 3328',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print('Each polynomial is just a list of 256 integers.')
print('Multiplying two of them (mod x²⁵⁶+1) is the core operation of Kyber.')
print(f'One multiply: {256*256:,} coefficient-level multiplications + {256*256:,} additions, reduced mod 3329.')

NameError: name 'np' is not defined

---
## Part 2 — The Bottleneck and the Fix

### Naïve polynomial multiplication is O(n²)
Multiplying two degree-255 polynomials the naïve way takes 256² ≈ 65,000 multiply-accumulate operations.
A full Kyber key exchange requires **12 such multiplications**. On an ARM Cortex-A9 (the CPU inside our board), this takes nearly **a full second**.

### The Number Theoretic Transform (NTT) brings it to O(n log n)
The NTT is the polynomial analogue of the Fast Fourier Transform (FFT).
It transforms a polynomial into a "frequency domain" where multiplication becomes pointwise — just 256 independent multiplications instead of 65,536 convolutions.
This reduces the operation count from **65,536** to roughly **3,500** — an 18× algorithmic speedup before touching the hardware.

### Our FPGA accelerator takes it further
We implemented the NTT in Vitis HLS (high-level synthesis) and synthesized it onto the PYNQ-Z2's programmable logic (PL):

| What | Where | Key detail |
|---|---|---|
| ARM Cortex-A9 | Processing System (PS) | Runs Python, orchestrates the KEM |
| NTT engine | Programmable Logic (PL) | Pipelined butterfly, II=2, 66 MHz |
| Polynomial BRAMs | On-chip SRAM (PL) | 3 × 256 × 32-bit, zero burst overhead |
| Control | AXI GPIO | PS pulses `ap_start`, polls `ap_done` |

The PS writes coefficients into on-chip BRAM, pulses a GPIO pin to start, and reads results back — no DDR memory, no DMA, no cache flushing.

---
## Part 3 — Live Benchmark

We run the same Kyber-512 key exchange twice:
- **Software path:** all polynomial multiplications in Python on the ARM CPU
- **Hardware path:** each multiplication dispatched to the FPGA via the C driver

In [ ]:
# Load the FPGA bitstream
from pynq import Overlay
ol = Overlay(os.path.join(BASE, 'bitstream/ntt_bd.bit'))
print('Bitstream loaded.')

# Build the driver if needed
driver_path = os.path.join(BASE, 'ps/ntt_driver')
if not os.path.exists(driver_path):
    print('Building ntt_driver...')
    subprocess.run(['make', '-C', os.path.join(BASE, 'ps')], check=True)
print(f'Driver ready: {driver_path}')

In [ ]:
SEED = 42
REPS = 10
a_test = [i % 3329 for i in range(256)]
b_test = [(i * 7 + 3) % 3329 for i in range(256)]

# --- Software KEM ---
print('Running software KEM...')
sw = run_kem(ntt_mul_sw, seed=SEED)
print(f'  KEM time:   {sw["time_ms"]:6.1f} ms   ({sw["mul_calls"]} poly multiplications)')

# --- Per-multiply SW ---
t0 = time.perf_counter()
for _ in range(REPS):
    ntt_mul_sw(a_test, b_test)
sw_mul_ms = (time.perf_counter() - t0) / REPS * 1000
print(f'  Per multiply: {sw_mul_ms:.2f} ms')

# --- Hardware warmup + KEM ---
print('\nWarming up hardware...')
ntt_mul_hw(a_test, b_test)   # first call starts the persistent driver process

print('Running hardware KEM...')
hw = run_kem(ntt_mul_hw, seed=SEED)
print(f'  KEM time:   {hw["time_ms"]:6.1f} ms   ({hw["mul_calls"]} poly multiplications)')

# --- Per-multiply HW ---
t0 = time.perf_counter()
for _ in range(REPS):
    ntt_mul_hw(a_test, b_test)
hw_mul_ms = (time.perf_counter() - t0) / REPS * 1000
print(f'  Per multiply: {hw_mul_ms:.3f} ms')

# --- Derived numbers ---
speedup_mul = sw_mul_ms / hw_mul_ms
speedup_kem = sw['time_ms'] / hw['time_ms']
sw_ntt_total = sw_mul_ms * sw['mul_calls']
hw_ntt_total = hw_mul_ms * hw['mul_calls']
sw_other     = sw['time_ms'] - sw_ntt_total
hw_other     = hw['time_ms'] - hw_ntt_total

print(f'\n  Per-multiply speedup: {speedup_mul:.0f}×')
print(f'  End-to-end KEM speedup: {speedup_kem:.1f}×')

In [ ]:
# ── Main benchmark chart ───────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 5))
gs  = GridSpec(1, 2, figure=fig, wspace=0.38)

# LEFT: per-multiply latency
ax1 = fig.add_subplot(gs[0])
vals   = [sw_mul_ms, hw_mul_ms]
labels = ['Python\n(ARM Cortex-A9)', 'FPGA\n(HLS NTT engine)']
bars   = ax1.bar(labels, vals, color=[SW_COLOR, HW_COLOR], width=0.45, zorder=3)
ax1.grid(axis='y', alpha=0.3, zorder=0)
ax1.set_ylabel('Latency (ms)', fontsize=11)
ax1.set_title('One polynomial multiplication', fontsize=12, fontweight='bold')
for bar, v in zip(bars, vals):
    ax1.text(bar.get_x() + bar.get_width()/2, v + max(vals)*0.02,
             f'{v:.2f} ms', ha='center', fontsize=11, fontweight='bold')
ax1.set_ylim(0, max(vals) * 1.22)
ax1.annotate('', xy=(1, hw_mul_ms + max(vals)*0.05),
             xytext=(0, sw_mul_ms * 0.5),
             arrowprops=dict(arrowstyle='->', color='green', lw=2))
ax1.text(0.5, (sw_mul_ms*0.5 + hw_mul_ms + max(vals)*0.05) / 2,
         f'{speedup_mul:.0f}× faster',
         ha='center', color='green', fontsize=13, fontweight='bold',
         transform=ax1.get_yaxis_transform())

# RIGHT: full KEM breakdown (stacked)
ax2 = fig.add_subplot(gs[1])
x    = np.array([0, 1])
ntt_times   = [sw_ntt_total, hw_ntt_total]
other_times = [max(0, sw_other), max(0, hw_other)]
b1 = ax2.bar(x, ntt_times,   color=[SW_COLOR, HW_COLOR], width=0.45,
             label='NTT multiplications', zorder=3, alpha=0.9)
b2 = ax2.bar(x, other_times, bottom=ntt_times,
             color=['#a0b8d8', '#f0a89a'], width=0.45,
             label='Sampling + additions (Python)', zorder=3, alpha=0.9)
totals = [sw['time_ms'], hw['time_ms']]
for xi, t in zip(x, totals):
    ax2.text(xi, t + max(totals)*0.02, f'{t:.0f} ms',
             ha='center', fontsize=11, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(['Python\n(ARM)', 'FPGA\n(HLS)'])
ax2.set_ylabel('Total time (ms)', fontsize=11)
ax2.set_title(f'Full Kyber-512 KEM  ({sw["mul_calls"]} multiplications)', fontsize=12, fontweight='bold')
ax2.set_ylim(0, max(totals) * 1.22)
ax2.grid(axis='y', alpha=0.3, zorder=0)
ax2.legend(loc='upper right', fontsize=9)
ax2.text(0.97, 0.92, f'{speedup_kem:.1f}× faster',
         transform=ax2.transAxes, ha='right', va='top',
         fontsize=13, color='green', fontweight='bold')

fig.suptitle('FPGA NTT Accelerator — Benchmark Results', fontsize=14, fontweight='bold', y=1.02)
plt.savefig('benchmark.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'The FPGA handles each NTT multiply in {hw_mul_ms*1000:.0f} µs.')
print(f'The stacked chart explains why {speedup_mul:.0f}× per-multiply → {speedup_kem:.1f}× KEM:')
print(f'  Sampling + additions take ~{sw_other:.0f} ms on both paths — the FPGA cannot help there.')

In [ ]:
# ── Throughput chart: operations per second ────────────────────────────────────
sw_muls_per_sec  = 1000 / sw_mul_ms
hw_muls_per_sec  = 1000 / hw_mul_ms
sw_kems_per_sec  = 1000 / sw['time_ms']
hw_kems_per_sec  = 1000 / hw['time_ms']

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, sw_v, hw_v, unit in zip(
    axes,
    [sw_muls_per_sec, sw_kems_per_sec],
    [hw_muls_per_sec, hw_kems_per_sec],
    ['NTT polynomial multiplications / second',
     'Complete Kyber-512 key exchanges / second']
):
    bars = ax.barh(['Python\n(ARM)', 'FPGA\n(HLS)'],
                   [sw_v, hw_v],
                   color=[SW_COLOR, HW_COLOR], height=0.4, zorder=3)
    ax.grid(axis='x', alpha=0.3, zorder=0)
    ax.set_xlabel(unit, fontsize=10)
    ax.set_title(unit, fontsize=10, fontweight='bold')
    for bar, v in zip(bars, [sw_v, hw_v]):
        ax.text(v + max(sw_v, hw_v)*0.01, bar.get_y() + bar.get_height()/2,
                f'{v:.1f}/s', va='center', fontsize=11, fontweight='bold')
    ax.set_xlim(0, max(sw_v, hw_v) * 1.25)

fig.suptitle('Throughput Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('throughput.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'FPGA handles {hw_muls_per_sec:.0f} polynomial multiplications per second')
print(f'vs {sw_muls_per_sec:.1f} in Python — {hw_muls_per_sec/sw_muls_per_sec:.0f}× higher throughput')

---
## Part 4 — Correctness Verification

Speed is meaningless if the answers are wrong.

We run both backends with the **same random seed** — this forces them to draw the same polynomial coefficients. If hardware and software produce identical shared secrets, the FPGA arithmetic is correct.

In [ ]:
assert sw['match'],               'SW KEM failed: Alice and Bob secrets differ'
assert hw['match'],               'HW KEM failed: Alice and Bob secrets differ'
assert sw['ss_alice'] == hw['ss_alice'], 'FPGA output differs from golden model'

fig, ax = plt.subplots(figsize=(10, 2.2))
ax.axis('off')
checks = [
    ('SW: Alice == Bob',              sw['match'],                    '✓'),
    ('HW: Alice == Bob',              hw['match'],                    '✓'),
    ('SW secret == HW secret',        sw['ss_alice'] == hw['ss_alice'], '✓'),
]
for i, (label, result, mark) in enumerate(checks):
    color = '#2ca02c' if result else '#d62728'
    ax.text(0.02, 0.8 - i*0.35, f'{mark}  {label}',
            transform=ax.transAxes, fontsize=13,
            color=color, fontweight='bold', va='top')

ax.text(0.6, 0.75, f'Shared secret (SW):\n{sw["ss_alice"].hex()[:32]}...', 
        transform=ax.transAxes, fontsize=9, family='monospace', color='#444')
ax.text(0.6, 0.35, f'Shared secret (HW):\n{hw["ss_alice"].hex()[:32]}...',
        transform=ax.transAxes, fontsize=9, family='monospace', color='#444')

ax.set_title('Correctness Checks', fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

print('All checks passed. FPGA output is bit-identical to the Python golden model.')

---
## Part 5 — Secure Communication Demo

The whole point of Kyber is to **establish a shared secret** between two parties who have never met — then use that secret to encrypt a message. Here's the full flow on hardware:

In [ ]:
def xor_encrypt(key: bytes, data: bytes) -> bytes:
    k = hashlib.sha256(key).digest()
    return bytes(d ^ k[i % 32] for i, d in enumerate(data))

message = b'Quantum computers cannot read this.'

# Use the hardware-derived secrets
ciphertext = xor_encrypt(hw['ss_bob'],   message)   # Bob encrypts
recovered  = xor_encrypt(hw['ss_alice'], ciphertext) # Alice decrypts

assert recovered == message

fig, axes = plt.subplots(1, 3, figsize=(13, 2.8))
steps = [
    ('Bob encrypts', message.decode(), '#2ca02c'),
    ('In transit (ciphertext)', ciphertext.hex()[:36]+'...', '#d62728'),
    ('Alice decrypts', recovered.decode(), '#2ca02c'),
]
for ax, (title, content, color) in zip(axes, steps):
    ax.set_facecolor('#f8f8f8')
    ax.text(0.5, 0.62, content, ha='center', va='center',
            fontsize=9 if len(content) > 30 else 11,
            family='monospace', color=color,
            fontweight='bold', transform=ax.transAxes,
            wrap=True)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(color)
        spine.set_linewidth(2)

fig.suptitle(f'Shared secret derived in {hw["time_ms"]:.0f} ms using FPGA-accelerated Kyber',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Key exchange: {hw["time_ms"]:.0f} ms  |  Message: {len(message)} bytes  |  Secure: post-quantum')

---
## Summary

| Metric | Software (ARM) | Hardware (FPGA) | Speedup |
|---|---|---|---|
| One NTT polynomial multiply | — ms | — ms | **—×** |
| Kyber-512 key exchange | — ms | — ms | **—×** |
| NTT muls / second | — | — | **—×** |

*(numbers filled in by the benchmark cell above)*

### What limits the KEM speedup?

The per-multiply speedup is much larger than the end-to-end KEM speedup because Kyber also involves polynomial **addition**, coefficient **sampling**, and **hashing** — none of which hit the FPGA. Amdahl's Law: if non-accelerated operations take time T, the maximum possible speedup is bounded by 1/(T/Total). A full C implementation of the KEM (no Python overhead) would show a speedup much closer to the per-multiply number.

### Architecture

```
ARM (PS)                     FPGA (PL)
─────────────────────        ─────────────────────────────
Python KEM logic      ──┐    ┌── BRAM_A (256 × 32-bit)
  writes a[], b[]     ──┼───►│   BRAM_B (256 × 32-bit)
  via AXI BRAM Ctrl   ──┘    │   BRAM_C (256 × 32-bit)
                             │
GPIO ap_start ───────────────►  ntt_top HLS IP
GPIO ap_done  ◄───────────────  NTT(a) → NTT(b) → mul → INTT(c)
                             │  Barrett reduction, pipelined II=2
reads c[] back ◄─────────────┘
```

**Target board:** PYNQ-Z2 (Xilinx Zynq-7000 XC7Z020, 85K LUTs, 240 DSPs, 4.9 Mb BRAM)  
**HLS tool:** Vitis HLS 2025.1  
**NTT algorithm:** FIPS 203 Algorithms 9/10 (Cooley-Tukey forward, Gentleman-Sande inverse)  
**Timing:** closed at 100 MHz with 2.3 ns WNS